In [2]:
from __future__ import annotations

import os
from collections.abc import Iterable
from typing import IO, Any, BinaryIO
import regex as re
import numpy.typing as npt
import torch
from jaxtyping import Bool, Float, Int
from torch import Tensor


def run_linear(
    d_in: int,
    d_out: int,
    weights: Float[Tensor, " d_out d_in"],
    in_features: Float[Tensor, " ... d_in"],
) -> Float[Tensor, " ... d_out"]:
    """
    Given the weights of a Linear layer, compute the transformation of a batched input.

    Args:
        in_dim (int): The size of the input dimension
        out_dim (int): The size of the output dimension
        weights (Float[Tensor, "d_out d_in"]): The linear weights to use
        in_features (Float[Tensor, "... d_in"]): The output tensor to apply the function to

    Returns:
        Float[Tensor, "... d_out"]: The transformed output of your linear module.
    """

    raise NotImplementedError


def run_embedding(
    vocab_size: int,
    d_model: int,
    weights: Float[Tensor, " vocab_size d_model"],
    token_ids: Int[Tensor, " ..."],
) -> Float[Tensor, " ... d_model"]:
    """
    Given the weights of an Embedding layer, get the embeddings for a batch of token ids.

    Args:
        vocab_size (int): The number of embeddings in the vocabulary
        d_model (int): The size of the embedding dimension
        weights (Float[Tensor, "vocab_size d_model"]): The embedding vectors to fetch from
        token_ids (Int[Tensor, "..."]): The set of token ids to fetch from the Embedding layer

    Returns:
        Float[Tensor, "... d_model"]: Batch of embeddings returned by your Embedding layer.
    """

    raise NotImplementedError


def run_swiglu(
    d_model: int,
    d_ff: int,
    w1_weight: Float[Tensor, " d_ff d_model"],
    w2_weight: Float[Tensor, " d_model d_ff"],
    w3_weight: Float[Tensor, " d_ff d_model"],
    in_features: Float[Tensor, " ... d_model"],
) -> Float[Tensor, " ... d_model"]:
    """Given the weights of a SwiGLU network, return
    the output of your implementation with these weights.

    Args:
        d_model (int): Dimensionality of the feedforward input and output.
        d_ff (int): Dimensionality of the up-project happening internally to your swiglu.
        w1_weight (Float[Tensor, "d_ff d_model"]): Stored weights for W1
        w2_weight (Float[Tensor, "d_model d_ff"]): Stored weights for W2
        w3_weight (Float[Tensor, "d_ff d_model"]): Stored weights for W3
        in_features (Float[Tensor, "... d_model"]): Input embeddings to the feed-forward layer.

    Returns:
        Float[Tensor, "... d_model"]: Output embeddings of the same shape as the input embeddings.
    """
    # Example:
    # If your state dict keys match, you can use `load_state_dict()`
    # swiglu.load_state_dict(weights)
    # You can also manually assign the weights
    # swiglu.w1.weight.data = w1_weight
    # swiglu.w2.weight.data = w2_weight
    # swiglu.w3.weight.data = w3_weight
    raise NotImplementedError


def run_scaled_dot_product_attention(
    Q: Float[Tensor, " ... queries d_k"],
    K: Float[Tensor, " ... keys d_k"],
    V: Float[Tensor, " ... keys d_v"],
    mask: Bool[Tensor, " ... queries keys"] | None = None,
) -> Float[Tensor, " ... queries d_v"]:
    """
    Given key (K), query (Q), and value (V) tensors, return
    the output of your scaled dot product attention implementation.

    Args:
        Q (Float[Tensor, " ... queries d_k"]): Query tensor
        K (Float[Tensor, " ... keys d_k"]): Key tensor
        V (Float[Tensor, " ... keys d_v"]): Values tensor
        mask (Bool[Tensor, " ... queries keys"] | None): Mask tensor
    Returns:
        Float[Tensor, " ... queries d_v"]: Output of SDPA
    """
    raise NotImplementedError


def run_multihead_self_attention(
    d_model: int,
    num_heads: int,
    q_proj_weight: Float[Tensor, " d_model d_model"],
    k_proj_weight: Float[Tensor, " d_model d_model"],
    v_proj_weight: Float[Tensor, " d_model d_model"],
    o_proj_weight: Float[Tensor, " d_model d_model"],
    in_features: Float[Tensor, " ... sequence_length d_model"],
) -> Float[Tensor, " ... sequence_length d_model"]:
    """
    Given the key, query, and value projection weights of a naive unbatched
    implementation of multi-head attention, return the output of an optimized batched
    implementation. This implementation should handle the key, query, and value projections
    for all heads in a single matrix multiply.
    This function should not use RoPE.
    See section 3.2.2 of Vaswani et al., 2017.

    Args:
        d_model (int): Dimensionality of the feedforward input and output.
        num_heads (int): Number of heads to use in multi-headed attention.
        max_seq_len (int): Maximum sequence length to pre-cache if your implementation does that.
        q_proj_weight (Float[Tensor, "d_model d_model"]): Weights for the Q projection
        k_proj_weight (Float[Tensor, "d_model d_model"]): Weights for the K projection
        v_proj_weight (Float[Tensor, "d_model d_model"]): Weights for the V projection
        o_proj_weight (Float[Tensor, "d_model d_model"]): Weights for the output projection
        in_features (Float[Tensor, "... sequence_length d_model"]): Tensor to run your implementation on.

    Returns:
        Float[Tensor, " ... sequence_length d_model"]: Tensor with the output of running your optimized, batched multi-headed attention
        implementation with the given QKV projection weights and input features.
    """
    raise NotImplementedError


def run_multihead_self_attention_with_rope(
    d_model: int,
    num_heads: int,
    max_seq_len: int,
    theta: float,
    q_proj_weight: Float[Tensor, " d_model d_model"],
    k_proj_weight: Float[Tensor, " d_model d_model"],
    v_proj_weight: Float[Tensor, " d_model d_model"],
    o_proj_weight: Float[Tensor, " d_model d_model"],
    in_features: Float[Tensor, " ... sequence_length d_model"],
    token_positions: Int[Tensor, " ... sequence_length"] | None = None,
) -> Float[Tensor, " ... sequence_length d_model"]:
    """
    Given the key, query, and value projection weights of a naive unbatched
    implementation of multi-head attention, return the output of an optimized batched
    implementation. This implementation should handle the key, query, and value projections
    for all heads in a single matrix multiply.
    This version of MHA should include RoPE.
    In this case, the RoPE embedding dimension must be the head embedding dimension (d_model // num_heads).
    See section 3.2.2 of Vaswani et al., 2017.

    Args:
        d_model (int): Dimensionality of the feedforward input and output.
        num_heads (int): Number of heads to use in multi-headed attention.
        max_seq_len (int): Maximum sequence length to pre-cache if your implementation does that.
        theta (float): RoPE parameter.
        q_proj_weight (Float[Tensor, "d_model d_model"]): Weights for the Q projection
        k_proj_weight (Float[Tensor, "d_model d_model"]): Weights for the K projection
        v_proj_weight (Float[Tensor, "d_model d_model"]): Weights for the V projection
        o_proj_weight (Float[Tensor, "d_model d_model"]): Weights for the output projection
        in_features (Float[Tensor, "... sequence_length d_model"]): Tensor to run your implementation on.
        token_positions (Int[Tensor, " ... sequence_length"] | None): Optional tensor with the positions of the tokens

    Returns:
        Float[Tensor, " ... sequence_length d_model"]: Tensor with the output of running your optimized, batched multi-headed attention
        implementation with the given QKV projection weights and input features.
    """
    raise NotImplementedError


def run_rope(
    d_k: int,
    theta: float,
    max_seq_len: int,
    in_query_or_key: Float[Tensor, " ... sequence_length d_k"],
    token_positions: Int[Tensor, " ... sequence_length"],
) -> Float[Tensor, " ... sequence_length d_k"]:
    """
    Run RoPE for a given input tensor.

    Args:
        d_k (int): Embedding dimension size for the query or key tensor.
        theta (float): RoPE parameter.
        max_seq_len (int): Maximum sequence length to pre-cache if your implementation does that.
        in_query_or_key (Float[Tensor, "... sequence_length d_k"]): Input tensor to run RoPE on.
        token_positions (Int[Tensor, "... sequence_length"]): Tensor of shape (batch_size, sequence_length) with the token positions
    Returns:
        Float[Tensor, " ... sequence_length d_k"]: Tensor with RoPEd input.
    """
    raise NotImplementedError


def run_transformer_block(
    d_model: int,
    num_heads: int,
    d_ff: int,
    max_seq_len: int,
    theta: float,
    weights: dict[str, Tensor],
    in_features: Float[Tensor, " batch sequence_length d_model"],
) -> Float[Tensor, " batch sequence_length d_model"]:
    """
    Given the weights of a pre-norm Transformer block and input features,
    return the output of running the Transformer block on the input features.

    This function should use RoPE.
    Depending on your implementation, you may simply need to pass the relevant args
    to your TransformerBlock constructor, or you may need to initialize your own RoPE
    class and pass that instead.

    Args:
        d_model (int): The dimensionality of the Transformer block input.
        num_heads (int): Number of heads to use in multi-headed attention. `d_model` must be
            evenly divisible by `num_heads`.
        d_ff (int): Dimensionality of the feed-forward inner layer.
        max_seq_len (int): Maximum sequence length to pre-cache if your implementation does that.
        theta (float): RoPE parameter.
        weights (dict[str, Tensor]):
            State dict of our reference implementation.
            The keys of this dictionary are:
            - `attn.q_proj.weight`
                The query projections for all `num_heads` attention heads.
                Shape is (d_model, d_model).
                The rows are ordered by matrices of shape (num_heads, d_k),
                so `attn.q_proj.weight == torch.cat([q_heads.0.weight, ..., q_heads.N.weight], dim=0)`.
            - `attn.k_proj.weight`
                The key projections for all `num_heads` attention heads.
                Shape is (d_model, d_model).
                The rows are ordered by matrices of shape (num_heads, d_k),
                so `attn.k_proj.weight == torch.cat([k_heads.0.weight, ..., k_heads.N.weight], dim=0)`.
            - `attn.v_proj.weight`
                The value projections for all `num_heads` attention heads.
                Shape is (d_model, d_model).
                The rows are ordered by matrices of shape (num_heads, d_v),
                so `attn.v_proj.weight == torch.cat([v_heads.0.weight, ..., v_heads.N.weight], dim=0)`.
            - `attn.output_proj.weight`
                Weight of the multi-head self-attention output projection
                Shape is (d_model, d_model).
            - `ln1.weight`
                Weights of affine transform for the first RMSNorm
                applied in the transformer block.
                Shape is (d_model,).
            - `ffn.w1.weight`
                Weight of the first linear transformation in the FFN.
                Shape is (d_ff, d_model).
            - `ffn.w2.weight`
                Weight of the second linear transformation in the FFN.
                Shape is (d_model, d_ff).
            - `ffn.w3.weight`
                Weight of the third linear transformation in the FFN.
                Shape is (d_ff, d_model).
            - `ln2.weight`
                Weights of affine transform for the second RMSNorm
                applied in the transformer block.
                Shape is (d_model,).
        in_features (Float[Tensor, "batch sequence_length d_model"]):
            Tensor to run your implementation on.

    Returns:
        Float[Tensor, "batch sequence_length d_model"] Tensor with the output of
        running the Transformer block on the input features while using RoPE.
    """
    raise NotImplementedError


def run_transformer_lm(
    vocab_size: int,
    context_length: int,
    d_model: int,
    num_layers: int,
    num_heads: int,
    d_ff: int,
    rope_theta: float,
    weights: dict[str, Tensor],
    in_indices: Int[Tensor, " batch_size sequence_length"],
) -> Float[Tensor, " batch_size sequence_length vocab_size"]:
    """Given the weights of a Transformer language model and input indices,
    return the output of running a forward pass on the input indices.

    This function should use RoPE.

    Args:
        vocab_size (int): The number of unique items in the output vocabulary to be predicted.
        context_length (int): The maximum number of tokens to process at once.
        d_model (int): The dimensionality of the model embeddings and sublayer outputs.
        num_layers (int): The number of Transformer layers to use.
        num_heads (int): Number of heads to use in multi-headed attention. `d_model` must be
            evenly divisible by `num_heads`.
        d_ff (int): Dimensionality of the feed-forward inner layer (section 3.3).
        rope_theta (float): The RoPE $\\Theta$ parameter.
        weights (dict[str, Tensor]):
            State dict of our reference implementation. {num_layers} refers to an
            integer between `0` and `num_layers - 1` (the layer index).
            The keys of this dictionary are:
            - `token_embeddings.weight`
                Token embedding matrix. Shape is (vocab_size, d_model).
            - `layers.{num_layers}.attn.q_proj.weight`
                The query projections for all `num_heads` attention heads.
                Shape is (num_heads * (d_model / num_heads), d_model).
                The rows are ordered by matrices of shape (num_heads, d_k),
                so `attn.q_proj.weight == torch.cat([q_heads.0.weight, ..., q_heads.N.weight], dim=0)`.
            - `layers.{num_layers}.attn.k_proj.weight`
                The key projections for all `num_heads` attention heads.
                Shape is (num_heads * (d_model / num_heads), d_model).
                The rows are ordered by matrices of shape (num_heads, d_k),
                so `attn.k_proj.weight == torch.cat([k_heads.0.weight, ..., k_heads.N.weight], dim=0)`.
            - `layers.{num_layers}.attn.v_proj.weight`
                The value projections for all `num_heads` attention heads.
                Shape is (num_heads * (d_model / num_heads), d_model).
                The rows are ordered by matrices of shape (num_heads, d_v),
                so `attn.v_proj.weight == torch.cat([v_heads.0.weight, ..., v_heads.N.weight], dim=0)`.
            - `layers.{num_layers}.attn.output_proj.weight`
                Weight of the multi-head self-attention output projection
                Shape is ((d_model / num_heads) * num_heads, d_model).
            - `layers.{num_layers}.ln1.weight`
                Weights of affine transform for the first RMSNorm
                applied in the transformer block.
                Shape is (d_model,).
            - `layers.{num_layers}.ffn.w1.weight`
                Weight of the first linear transformation in the FFN.
                Shape is (d_ff, d_model).
            - `layers.{num_layers}.ffn.w2.weight`
                Weight of the second linear transformation in the FFN.
                Shape is (d_model, d_ff).
            - `layers.{num_layers}.ffn.w3.weight`
                Weight of the third linear transformation in the FFN.
                Shape is (d_ff, d_model).
            - `layers.{num_layers}.ln2.weight`
                Weights of affine transform for the second RMSNorm
                applied in the transformer block.
                Shape is (d_model,).
            - `ln_final.weight`
                Weights of affine transform for RMSNorm applied to the output of the final transformer block.
                Shape is (d_model, ).
            - `lm_head.weight`
                Weights of the language model output embedding.
                Shape is (vocab_size, d_model).
        in_indices (Int[Tensor, "batch_size sequence_length"]) Tensor with input indices to run the language model on. Shape is (batch_size, sequence_length), where
            `sequence_length` is at most `context_length`.

    Returns:
        Float[Tensor, "batch_size sequence_length vocab_size"]: Tensor with the predicted unnormalized
        next-word distribution for each token.
    """
    raise NotImplementedError


def run_rmsnorm(
    d_model: int,
    eps: float,
    weights: Float[Tensor, " d_model"],
    in_features: Float[Tensor, " ... d_model"],
) -> Float[Tensor, " ... d_model"]:
    """Given the weights of a RMSNorm affine transform,
    return the output of running RMSNorm on the input features.

    Args:
        d_model (int): The dimensionality of the RMSNorm input.
        eps: (float): A value added to the denominator for numerical stability.
        weights (Float[Tensor, "d_model"]): RMSNorm weights.
        in_features (Float[Tensor, "... d_model"]): Input features to run RMSNorm on. Can have arbitrary leading
            dimensions.

    Returns:
        Float[Tensor,"... d_model"]: Tensor of with the same shape as `in_features` with the output of running
        RMSNorm of the `in_features`.
    """
    raise NotImplementedError


def run_silu(in_features: Float[Tensor, " ..."]) -> Float[Tensor, " ..."]:
    """Given a tensor of inputs, return the output of applying SiLU
    to each element.

    Args:
        in_features(Float[Tensor, "..."]): Input features to run SiLU on. Shape is arbitrary.

    Returns:
        Float[Tensor,"..."]: of with the same shape as `in_features` with the output of applying
        SiLU to each element.
    """
    raise NotImplementedError


def run_get_batch(
    dataset: npt.NDArray, batch_size: int, context_length: int, device: str
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Given a dataset (a 1D numpy array of integers) and a desired batch size and
    context length, sample language modeling input sequences and their corresponding
    labels from the dataset.

    Args:
        dataset (np.array): 1D numpy array of integer token IDs in the dataset.
        batch_size (int): Desired batch size to sample.
        context_length (int): Desired context length of each sampled example.
        device (str): PyTorch device string (e.g., 'cpu' or 'cuda:0') indicating the device
            to place the sampled input sequences and labels on.

    Returns:
        Tuple of torch.LongTensors of shape (batch_size, context_length). The first tuple item
        is the sampled input sequences, and the second tuple item is the corresponding
        language modeling labels.
    """
    raise NotImplementedError


def run_softmax(in_features: Float[Tensor, " ..."], dim: int) -> Float[Tensor, " ..."]:
    """
    Given a tensor of inputs, return the output of softmaxing the given `dim`
    of the input.

    Args:
        in_features (Float[Tensor, "..."]): Input features to softmax. Shape is arbitrary.
        dim (int): Dimension of the `in_features` to apply softmax to.

    Returns:
        Float[Tensor, "..."]: Tensor of with the same shape as `in_features` with the output of
        softmax normalizing the specified `dim`.
    """
    raise NotImplementedError


def run_cross_entropy(
    inputs: Float[Tensor, " batch_size vocab_size"], targets: Int[Tensor, " batch_size"]
) -> Float[Tensor, ""]:
    """Given a tensor of inputs and targets, compute the average cross-entropy
    loss across examples.

    Args:
        inputs (Float[Tensor, "batch_size vocab_size"]): inputs[i][j] is the
            unnormalized logit of jth class for the ith example.
        targets (Int[Tensor, "batch_size"]): Tensor of shape (batch_size,) with the index of the correct class.
            Each value must be between 0 and `num_classes - 1`.

    Returns:
        Float[Tensor, ""]: The average cross-entropy loss across examples.
    """
    raise NotImplementedError


def run_gradient_clipping(parameters: Iterable[torch.nn.Parameter], max_l2_norm: float) -> None:
    """Given a set of parameters, clip their combined gradients to have l2 norm at most max_l2_norm.

    Args:
        parameters (Iterable[torch.nn.Parameter]): collection of trainable parameters.
        max_l2_norm (float): a positive value containing the maximum l2-norm.

    The gradients of the parameters (parameter.grad) should be modified in-place.
    """
    raise NotImplementedError


def get_adamw_cls() -> Any:
    """
    Returns a torch.optim.Optimizer that implements AdamW.
    """
    raise NotImplementedError


def run_get_lr_cosine_schedule(
    it: int,
    max_learning_rate: float,
    min_learning_rate: float,
    warmup_iters: int,
    cosine_cycle_iters: int,
):
    """
    Given the parameters of a cosine learning rate decay schedule (with linear
    warmup) and an iteration number, return the learning rate at the given
    iteration under the specified schedule.

    Args:
        it (int): Iteration number to get learning rate for.
        max_learning_rate (float): alpha_max, the maximum learning rate for
            cosine learning rate schedule (with warmup).
        min_learning_rate (float): alpha_min, the minimum / final learning rate for
            the cosine learning rate schedule (with warmup).
        warmup_iters (int): T_w, the number of iterations to linearly warm-up
            the learning rate.
        cosine_cycle_iters (int): T_c, the number of cosine annealing iterations.

    Returns:
        Learning rate at the given iteration under the specified schedule.
    """
    raise NotImplementedError


def run_save_checkpoint(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    iteration: int,
    out: str | os.PathLike | BinaryIO | IO[bytes],
):
    """
    Given a model, optimizer, and an iteration number, serialize them to disk.

    Args:
        model (torch.nn.Module): Serialize the state of this model.
        optimizer (torch.optim.Optimizer): Serialize the state of this optimizer.
        iteration (int): Serialize this value, which represents the number of training iterations
            we've completed.
        out (str | os.PathLike | BinaryIO | IO[bytes]): Path or file-like object to serialize the model, optimizer, and iteration to.
    """
    raise NotImplementedError


def run_load_checkpoint(
    src: str | os.PathLike | BinaryIO | IO[bytes],
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
) -> int:
    """
    Given a serialized checkpoint (path or file-like object), restore the
    serialized state to the given model and optimizer.
    Return the number of iterations that we previously serialized in
    the checkpoint.

    Args:
        src (str | os.PathLike | BinaryIO | IO[bytes]): Path or file-like object to serialized checkpoint.
        model (torch.nn.Module): Restore the state of this model.
        optimizer (torch.optim.Optimizer): Restore the state of this optimizer.
    Returns:
        int: the previously-serialized number of iterations.
    """
    raise NotImplementedError


def get_tokenizer(
    vocab: dict[int, bytes],
    merges: list[tuple[bytes, bytes]],
    special_tokens: list[str] | None = None,
) -> Any:
    """Given a vocabulary, a list of merges, and a list of special tokens,
    return a BPE tokenizer that uses the provided vocab, merges, and special tokens.

    Args:
        vocab (dict[int, bytes]): The tokenizer vocabulary, a mapping from int (token ID in the vocabulary)
            to bytes (token bytes)
        merges (list[tuple[bytes, bytes]]): BPE merges. Each list item is a tuple of bytes (<token1>, <token2>),
            representing that <token1> was merged with <token2>.
            Merges are ordered by order of creation.
        special_tokens (list[str] | None): A list of string special tokens for the tokenizer. These strings will never
            be split into multiple tokens, and will always be kept as a single token.

    Returns:
        A BPE tokenizer that uses the provided vocab, merges, and special tokens.
    """
    raise NotImplementedError

def rebuild(word, best_pair, new_token):
    result = []                     # 收集新词的列表
    i = 0
    while i < len(word):
        if i + 1 < len(word) and word[i:i+2] == best_pair:   # 相邻两个正好是要合并的
            result.append(new_token)   # 收下合成的新 token
            i += 2                     # 跳过这两个
        else:
            result.append(word[i])     # 收下当前 token
            i += 1                     # 走一步
    return tuple(result)               # 新词


def run_train_bpe(
    input_path: str | os.PathLike,
    vocab_size: int,
    special_tokens: list[str],
    **kwargs,
) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
    """Given the path to an input corpus, run train a BPE tokenizer and
    output its vocabulary and merges.

    Args:
        input_path (str | os.PathLike): Path to BPE tokenizer training data.
        vocab_size (int): Total number of items in the tokenizer's vocabulary (including special tokens).
        special_tokens (list[str]): A list of string special tokens to be added to the tokenizer vocabulary.
            These strings will never be split into multiple tokens, and will always be
            kept as a single token. If these special tokens occur in the `input_path`,
            they are treated as any other string.

    Returns:
        tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
            vocab:
                The trained tokenizer vocabulary, a mapping from int (token ID in the vocabulary)
                to bytes (token bytes)
            merges:
                BPE merges. Each list item is a tuple of bytes (<token1>, <token2>),
                representing that <token1> was merged with <token2>.
                Merges are ordered by order of creation.
    """
    # ========= 初始化 vocab =========
    vocab = {}                                   # dict[int, bytes]
    for i, st in enumerate(special_tokens):
        vocab[len(vocab)] = st.encode("utf-8")   # 字符串.encode() → bytes
    for b in range(256):                         # 256 个字节
        vocab[len(vocab)] = bytes([b])
    with open(input_path) as f:
        content=f.read()
        PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
        subcontent =re.findall(PAT,content)
        chardict={c:0 for c in subcontent}
        for c in subcontent:
            chardict[c]+=1
        merge_list=[]
        bytesdict= {
                tuple(bytes([x]) for x in word.encode("utf-8")): freq
                for word, freq in chardict.items()
            }
        while len(vocab) < vocab_size:    
            # 对byte进行数数
            pair_counts={}
            for word,freq in bytesdict.items():
                for i in range(len(word)-1):
                    word_pair=(word[i],word[i+1])
                    pair_counts[word_pair]=pair_counts.get(word_pair,0)+freq
            best_pair=max(pair_counts.items(),key=lambda x:(x[1],x[0]))[0]#找到数值最大的字节流
            newtoken=best_pair[0]+best_pair[1]
            merge_list.append(best_pair)
            new_bytes_dict={}
        
            for word, freq in bytesdict.items():
                merged_word = rebuild(word, best_pair, newtoken)   # ← 上面那个函数
                new_bytes_dict[merged_word] = new_bytes_dict.get(merged_word, 0) + freq
            bytesdict = new_bytes_dict #完成字典更新
            vocab[len(vocab)]=newtoken
        return vocab,merge_list
            




In [3]:
# ========= 初始化 vocab =========
vocab = {}                                   # dict[int, bytes]
vocab_size=258
for b in range(256):                         # 256 个字节
    vocab[len(vocab)] = bytes([b])

    content=""" la la la udsd sad isd la """
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    subcontent =re.findall(PAT,content)
    chardict={c:0 for c in subcontent}
    for c in subcontent:
        chardict[c]+=1
    merge_list=[]
    bytesdict= {
            tuple(bytes([x]) for x in word.encode("utf-8")): freq
    for word, freq in chardict.items()
    }
print(subcontent)
print(bytesdict)
pair_counts={}
for word,freq in bytesdict.items():
    for i in range(len(word)-1):
        word_pair=(word[i],word[i+1])
        pair_counts[word_pair]=pair_counts.get(word_pair,0)+freq
print("pair_counts")
print(pair_counts)# 对byte进行数数
while len(vocab) < vocab_size:   
    best_pair=max(pair_counts.items(),key=lambda x:(x[1],x[0]))[0]#找到数值最大的字节流
    newtoken=best_pair[0]+best_pair[1] 
    merge_list.append(best_pair)
    new_bytes_dict={}
    new_pair_counts={}
    for word, freq in bytesdict.items():
        print("word")
        print(word)
        merged_word = rebuild(word, best_pair, newtoken)   # ← 上面那个函数
        print("merged_word")
        print(merged_word)
        print("next")
        new_bytes_dict[merged_word] = new_bytes_dict.get(merged_word, 0) + freq
        
        if merged_word!= word:
            for i in range(len(word)-1):
                # 销毁键，更新键，减少对字典的遍历
                word_pair=(word[i],word[i+1])
                pair_counts[word_pair]=pair_counts.get(word_pair,0)-freq
            for i in range(len(merged_word)-1):
                merge_word_pair=(merged_word[i],merged_word[i+1])
                pair_counts[merge_word_pair]=pair_counts.get(merge_word_pair,0)-freq
    bytesdict = new_bytes_dict #完成字典更新
    vocab[len(vocab)]=newtoken


[' la', ' la', ' la', ' udsd', ' sad', ' isd', ' la', ' ']
{(b' ', b'l', b'a'): 4, (b' ', b'u', b'd', b's', b'd'): 1, (b' ', b's', b'a', b'd'): 1, (b' ', b'i', b's', b'd'): 1, (b' ',): 1}
pair_counts
{(b' ', b'l'): 4, (b'l', b'a'): 4, (b' ', b'u'): 1, (b'u', b'd'): 1, (b'd', b's'): 1, (b's', b'd'): 2, (b' ', b's'): 1, (b's', b'a'): 1, (b'a', b'd'): 1, (b' ', b'i'): 1, (b'i', b's'): 1}
word
(b' ', b'l', b'a')
merged_word
(b' ', b'la')
next
word
(b' ', b'u', b'd', b's', b'd')
merged_word
(b' ', b'u', b'd', b's', b'd')
next
word
(b' ', b's', b'a', b'd')
merged_word
(b' ', b's', b'a', b'd')
next
word
(b' ', b'i', b's', b'd')
merged_word
(b' ', b'i', b's', b'd')
next
word
(b' ',)
merged_word
(b' ',)
next
word
(b' ', b'la')
merged_word
(b' ', b'la')
next
word
(b' ', b'u', b'd', b's', b'd')
merged_word
(b' ', b'u', b'd', b'sd')
next
word
(b' ', b's', b'a', b'd')
merged_word
(b' ', b's', b'a', b'd')
next
word
(b' ', b'i', b's', b'd')
merged_word
(b' ', b'i', b'sd')
next
word
(b' ',)
merged_wor

In [7]:
# ========= 初始化 vocab =========
vocab = {}                                   # dict[int, bytes]
vocab_size=258
for b in range(256):                         # 256 个字节
    vocab[len(vocab)] = bytes([b])

    content=""" la la la udsd sad isd la """
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    subcontent =re.findall(PAT,content)
    chardict={c:0 for c in subcontent}
    for c in subcontent:
        chardict[c]+=1
    merge_list=[]
    bytesdict= {
            tuple(bytes([x]) for x in word.encode("utf-8")): freq
    for word, freq in chardict.items()
    }
print(subcontent)
print(bytesdict)
while len(vocab) < vocab_size:    
    pair_counts={}
    for word,freq in bytesdict.items():
        for i in range(len(word)-1):
            word_pair=(word[i],word[i+1])
            pair_counts[word_pair]=pair_counts.get(word_pair,0)+freq
    print("pair_counts")
    print(pair_counts)# 对byte进行数数
    best_pair=max(pair_counts.items(),key=lambda x:(x[1],x[0]))[0]#找到数值最大的字节流
    print("best_pair")
    print(best_pair)
    newtoken=best_pair[0]+best_pair[1]
    print(newtoken)
    merge_list.append(best_pair)
    new_bytes_dict={}
    new_pair_counts={}
    for word, freq in bytesdict.items():
        print("word")
        print(word)
        merged_word = rebuild(word, best_pair, newtoken)   # ← 上面那个函数
        print("merged_word")
        print(merged_word)
        print("next")
        new_bytes_dict[merged_word] = new_bytes_dict.get(merged_word, 0) + freq        
    bytesdict = new_bytes_dict #完成字典更新
    vocab[len(vocab)]=newtoken


[' la', ' la', ' la', ' udsd', ' sad', ' isd', ' la', ' ']
{(b' ', b'l', b'a'): 4, (b' ', b'u', b'd', b's', b'd'): 1, (b' ', b's', b'a', b'd'): 1, (b' ', b'i', b's', b'd'): 1, (b' ',): 1}
pair_counts
{(b' ', b'l'): 4, (b'l', b'a'): 4, (b' ', b'u'): 1, (b'u', b'd'): 1, (b'd', b's'): 1, (b's', b'd'): 2, (b' ', b's'): 1, (b's', b'a'): 1, (b'a', b'd'): 1, (b' ', b'i'): 1, (b'i', b's'): 1}
best_pair
(b'l', b'a')
b'la'
word
(b' ', b'l', b'a')
merged_word
(b' ', b'la')
next
word
(b' ', b'u', b'd', b's', b'd')
merged_word
(b' ', b'u', b'd', b's', b'd')
next
word
(b' ', b's', b'a', b'd')
merged_word
(b' ', b's', b'a', b'd')
next
word
(b' ', b'i', b's', b'd')
merged_word
(b' ', b'i', b's', b'd')
next
word
(b' ',)
merged_word
(b' ',)
next
pair_counts
{(b' ', b'la'): 4, (b' ', b'u'): 1, (b'u', b'd'): 1, (b'd', b's'): 1, (b's', b'd'): 2, (b' ', b's'): 1, (b's', b'a'): 1, (b'a', b'd'): 1, (b' ', b'i'): 1, (b'i', b's'): 1}
best_pair
(b' ', b'la')
b' la'
word
(b' ', b'la')
merged_word
(b' la',)
next
w

In [ ]:
# toy_index.py —— 索引版 merge 的迷你演示（整数版，逻辑与真实代码一致）
# 真实代码里：词 = bytes 的元组（如 ('a','b')）；这里用整数元组代替，机制一模一样

words = {(1, 2, 3): 3, (1, 2): 2, (4, 5): 5}      # 模拟 bytesdict：词元组 -> 频次

def rebuild(word, best_pair, new_token):
    """把 word 里所有相邻的 best_pair 替换成 new_token"""
    out, i = [], 0
    while i < len(word):
        if i + 1 < len(word) and (word[i], word[i + 1]) == best_pair:
            out.append(new_token)
            i += 2
        else:
            out.append(word[i])
            i += 1
    return tuple(out)

# ① 初始化：建倒排表 pair_words（顺带建 pair_counts）
pair_counts = {}
pair_words = {}
for word, freq in words.items():
    for i in range(len(word) - 1):
        p = (word[i], word[i + 1])
        pair_counts[p] = pair_counts.get(p, 0) + freq
        pair_words.setdefault(p, {})[word] = freq
        

print("初始 pair_counts:", pair_counts)
print("初始 pair_words :", pair_words)

# ② 合并一轮：best = (1, 2) → 新 token = 12
best_pair = (1, 2)
new_token = 12
affected = list(pair_words[best_pair].items())     # 只拿含 best_pair 的词
print("\n受影响的词:", affected)

for word, freq in affected:
    # 撤旧：把 word 从它所有相邻 pair 的登记里注销，pair_counts 同步减
    for i in range(len(word) - 1):
        p = (word[i], word[i + 1])
        pair_counts[p] = pair_counts.get(p, 0) - freq   # 用 get 兜底，键可能不存在
        pw = pair_words.get(p)
        if pw is not None:
            pw.pop(word, None)
            if not pw:
                del pair_words[p]

    merged_word = rebuild(word, best_pair, new_token)

    # 改 bytesdict：删旧词、加新词（可能已有同名词，累加）
    del words[word]
    words[merged_word] = words.get(merged_word, 0) + freq

    # 加新：把 merged_word 登记进它所有相邻 pair，pair_counts 同步加
    for i in range(len(merged_word) - 1):
        p = (merged_word[i], merged_word[i + 1])
        pair_counts[p] = pair_counts.get(p, 0) + freq
        pw2 = pair_words.setdefault(p, {})
        pw2[merged_word] = pw2.get(merged_word, 0) + freq

print("\n合并后 words      :", words)
print("合并后 pair_counts:", pair_counts)
print("合并后 pair_words :", pair_words)
print("\n自检 pair_counts[(1,2)]==0:", pair_counts[(1, 2)] == 0)


{(1, 2): 3}
{(1, 2): {(1, 2, 3): 3}}
{(1, 2): 3, (2, 3): 3}
{(1, 2): {(1, 2, 3): 3}, (2, 3): {(1, 2, 3): 3}}
{(1, 2): 5, (2, 3): 3}
{(1, 2): {(1, 2, 3): 3, (1, 2): 2}, (2, 3): {(1, 2, 3): 3}}
{(1, 2): 5, (2, 3): 3, (4, 5): 5}
{(1, 2): {(1, 2, 3): 3, (1, 2): 2}, (2, 3): {(1, 2, 3): 3}, (4, 5): {(4, 5): 5}}
初始 pair_counts: {(1, 2): 5, (2, 3): 3, (4, 5): 5}
初始 pair_words : {(1, 2): {(1, 2, 3): 3, (1, 2): 2}, (2, 3): {(1, 2, 3): 3}, (4, 5): {(4, 5): 5}}

受影响的词: [((1, 2, 3), 3), ((1, 2), 2)]

合并后 words      : {(4, 5): 5, (12, 3): 3, (12,): 2}
合并后 pair_counts: {(1, 2): 0, (2, 3): 0, (4, 5): 5, (12, 3): 3}
合并后 pair_words : {(4, 5): {(4, 5): 5}, (12, 3): {(12, 3): 3}}

自检 pair_counts[(1,2)]==0: True
